# Bag-of-words & TF-IDF — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def softmax(x, axis=-1):
    x=np.asarray(x, dtype=float); x=x-x.max(axis=axis, keepdims=True); e=np.exp(x); return e/e.sum(axis=axis, keepdims=True)
def sigmoid(x): return 1/(1+np.exp(-np.asarray(x, dtype=float)))
def normalize(v):
    v=np.asarray(v, dtype=float); n=np.linalg.norm(v); return v/(n if n else 1)
def edit_distance(a,b):
    D=np.zeros((len(a)+1,len(b)+1), dtype=int); D[:,0]=np.arange(len(a)+1); D[0,:]=np.arange(len(b)+1)
    for i in range(1,len(a)+1):
        for j in range(1,len(b)+1):
            D[i,j]=min(D[i-1,j]+1, D[i,j-1]+1, D[i-1,j-1]+(a[i-1]!=b[j-1]))
    return D
print('setup ok')

## Represent documents by what words occur, then downweight words that occur everywhere
The five examples isolate counts, binary presence, term frequency, inverse document frequency, and cosine retrieval under TF-IDF.

In [ ]:
docs=['cat sat','cat ate fish','dog sat']; vocab=sorted(set(' '.join(docs).split()))
X=np.array([[d.split().count(w) for w in vocab] for d in docs])
plt.figure(figsize=(6,3)); plt.imshow(X, cmap='Blues'); plt.xticks(range(len(vocab)),vocab); plt.yticks(range(3),['d0','d1','d2']); plt.title('Bag-of-words count matrix')
assert X.shape==(3,5) and X[0,vocab.index('cat')]==1

In [ ]:
binX=(X>0).astype(int)
plt.figure(figsize=(6,3)); plt.imshow(binX, cmap='Greens'); plt.xticks(range(len(vocab)),vocab); plt.yticks(range(3),['d0','d1','d2']); plt.title('Binary BoW keeps presence, drops repetition')
assert binX.sum()==7

In [ ]:
d='cat cat sat'.split(); tf=np.array([d.count(w)/len(d) for w in vocab])
plt.figure(figsize=(6,3)); plt.bar(vocab,tf); plt.title('Term frequency normalizes by document length')
assert abs(tf[vocab.index('cat')]-2/3)<1e-9

In [ ]:
df=np.array([(X[:,j]>0).sum() for j in range(len(vocab))]); idf=np.log((1+len(docs))/(1+df))+1
plt.figure(figsize=(6,3)); plt.bar(vocab,idf); plt.title('IDF downweights common words')
assert idf[vocab.index('fish')]>idf[vocab.index('cat')]

In [ ]:
tfidf=X*idf; q=np.array([1 if w in ['cat','fish'] else 0 for w in vocab])*idf
sims=tfidf@q/(np.linalg.norm(tfidf,axis=1)*np.linalg.norm(q))
plt.figure(figsize=(6,3)); plt.bar(['d0','d1','d2'],sims); plt.title('TF-IDF cosine retrieves the topical document')
assert int(np.argmax(sims))==1